In [2]:
# !pip install transformers datasets torch

In [3]:
from datasets import load_dataset
from transformers import AutoTokenizer
from torch.utils.data import DataLoader
import torch
from collections import Counter

d:\T\MS\MLOps\lab5\LLM_Data_Pipeline\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# 1. Load AG News dataset (news article classification — 4 categories)
# Replacing wikitext-2 with ag_news for a more practical, real-world text dataset
dataset = load_dataset("ag_news", split="train")
print(f"Number of examples in dataset: {len(dataset)}")
print(f"Features: {dataset.features}")

d:\T\MS\MLOps\lab5\LLM_Data_Pipeline\venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\tejas\.cache\huggingface\hub\datasets--ag_news. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating test split: 100%|██████████| 7600/7600 [00:00<00:00, 1336157.54 examples/s]

Number of examples in dataset: 120000
Features: {'text': Value('string'), 'label': ClassLabel(names=['World', 'Sports', 'Business', 'Sci/Tech'])}


In [5]:
# 2. Exploratory Data Analysis (EDA)

# Label mapping for AG News
label_names = ["World", "Sports", "Business", "Sci/Tech"]

# 2a. Sample texts per category
print("=== Sample texts per category ===")
for label_id, label_name in enumerate(label_names):
    sample = next(ex for ex in dataset if ex["label"] == label_id)
    print(f"\n[{label_name}]: {sample['text'][:120]}...")

# 2b. Label distribution
print("\n=== Label distribution ===")
label_counts = Counter(dataset["label"])
for label_id, count in sorted(label_counts.items()):
    print(f"  {label_names[label_id]}: {count} examples ({count/len(dataset)*100:.1f}%)")

# 2c. Text length distribution (in characters)
lengths = [len(ex["text"]) for ex in dataset]
print(f"\n=== Text length (chars) ===")
print(f"  Min:    {min(lengths)}")
print(f"  Max:    {max(lengths)}")
print(f"  Mean:   {sum(lengths)/len(lengths):.1f}")

=== Sample texts per category ===

[World]: Venezuelans Vote Early in Referendum on Chavez Rule (Reuters) Reuters - Venezuelans turned out early\and in large number...

[Sports]: Phelps, Thorpe Advance in 200 Freestyle (AP) AP - Michael Phelps took care of qualifying for the Olympic 200-meter frees...

[Business]: Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\band of ultra-cynics,...

[Sci/Tech]: 'Madden,' 'ESPN' Football Score in Different Ways (Reuters) Reuters - Was absenteeism a little high\on Tuesday among the...

=== Label distribution ===
  World: 30000 examples (25.0%)
  Sports: 30000 examples (25.0%)
  Business: 30000 examples (25.0%)
  Sci/Tech: 30000 examples (25.0%)

=== Text length (chars) ===
  Min:    100
  Max:    1012
  Mean:   236.5


In [6]:
# 3. Initialize tokenizer
# Using distilgpt2 instead of gpt2 — same tokenizer vocabulary but lighter weight
tokenizer = AutoTokenizer.from_pretrained("distilgpt2")
tokenizer.pad_token = tokenizer.eos_token  # distilgpt2 also has no pad token by default
print(f"Vocab size: {tokenizer.vocab_size}")

d:\T\MS\MLOps\lab5\LLM_Data_Pipeline\venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\tejas\.cache\huggingface\hub\models--distilgpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Vocab size: 50257


In [7]:
# 4. Tokenize the dataset using batched processing
def tokenize_function(examples):
    return tokenizer(examples["text"], return_special_tokens_mask=False)

tokenized_ds = dataset.map(tokenize_function, batched=True, remove_columns=["text"])
# Columns remaining: input_ids, attention_mask, label
print(f"Columns after tokenization: {tokenized_ds.column_names}")
print(f"Sample token IDs (first 20): {tokenized_ds[0]['input_ids'][:20]}")

Map: 100%|██████████| 120000/120000 [00:03<00:00, 38998.68 examples/s]

Columns after tokenization: ['label', 'input_ids', 'attention_mask']
Sample token IDs (first 20): [22401, 520, 13, 15682, 30358, 5157, 20008, 262, 2619, 357, 12637, 8, 8428, 532, 10073, 12, 7255, 364, 11, 5007]


In [8]:
# 5. Slice into fixed-length training sequences
# Using block_size=256 (increased from 128) to capture more context per training sample
block_size = 256

def group_texts(examples):
    # Concatenate all token sequences
    concatenated_inputs = sum(examples["input_ids"], [])
    concatenated_masks = sum(examples["attention_mask"], [])
    # Keep only complete blocks
    total_len = (len(concatenated_inputs) // block_size) * block_size
    concatenated_inputs = concatenated_inputs[:total_len]
    concatenated_masks = concatenated_masks[:total_len]
    # Split into chunks of block_size
    result_input_ids = [concatenated_inputs[i:i+block_size] for i in range(0, total_len, block_size)]
    result_masks = [concatenated_masks[i:i+block_size] for i in range(0, total_len, block_size)]
    return {"input_ids": result_input_ids, "attention_mask": result_masks}

# Note: grouping discards the per-example labels (blocks span multiple articles)
# We keep the original tokenized_ds with labels for classification use in the DataLoader below
lm_ds = tokenized_ds.remove_columns(["label"]).map(group_texts, batched=True, batch_size=1000)
print(f"LM training sequences (block_size={block_size}): {len(lm_ds)}")

Map: 100%|██████████| 120000/120000 [00:18<00:00, 6499.36 examples/s]

LM training sequences (block_size=256): 24361


In [9]:
# 6. Create a DataLoader for LM training
def collate_fn(batch):
    input_ids = torch.tensor([example["input_ids"] for example in batch], dtype=torch.long)
    # For causal LM, labels = input_ids (the model predicts the next token at each position)
    return {"input_ids": input_ids, "labels": input_ids.clone()}

train_loader = DataLoader(lm_ds, batch_size=8, shuffle=True, collate_fn=collate_fn)
print(f"Total batches in DataLoader: {len(train_loader)}")

Total batches in DataLoader: 3046


In [10]:
# 7. Verify a batch
for batch in train_loader:
    print(f"input_ids shape : {batch['input_ids'].shape}")
    print(f"labels shape    : {batch['labels'].shape}")
    print(f"Sample token IDs: {batch['input_ids'][0][:20]}")
    break

input_ids shape : torch.Size([8, 256])
labels shape    : torch.Size([8, 256])
Sample token IDs: tensor([  357, 17449,     8,  8916,   532,  1052,  3942,  5342,   531,   257,
         1524,  2420,    12,  2070,   973,   287,   262,  3685,    12, 46330])


In [11]:
# 8. Bonus: DataLoader for classification (preserving labels)
# Pad/truncate each example individually to block_size for classification use
def classification_collate_fn(batch):
    # Truncate or pad each example to block_size
    input_ids = []
    attention_masks = []
    labels = []
    for ex in batch:
        ids = ex["input_ids"][:block_size]
        mask = ex["attention_mask"][:block_size]
        # Pad if shorter than block_size
        pad_len = block_size - len(ids)
        ids = ids + [tokenizer.pad_token_id] * pad_len
        mask = mask + [0] * pad_len
        input_ids.append(ids)
        attention_masks.append(mask)
        labels.append(ex["label"])
    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "attention_mask": torch.tensor(attention_masks, dtype=torch.long),
        "labels": torch.tensor(labels, dtype=torch.long),
    }

classification_loader = DataLoader(tokenized_ds, batch_size=8, shuffle=True, collate_fn=classification_collate_fn)

for batch in classification_loader:
    print(f"input_ids shape     : {batch['input_ids'].shape}")
    print(f"attention_mask shape: {batch['attention_mask'].shape}")
    print(f"labels shape        : {batch['labels'].shape}")
    print(f"Label values        : {batch['labels']}")
    break

input_ids shape     : torch.Size([8, 256])
attention_mask shape: torch.Size([8, 256])
labels shape        : torch.Size([8])
Label values        : tensor([3, 1, 3, 1, 1, 3, 2, 2])
